# 3W Dataset Inspection

**Goal:** understand the structure, labels and composition of the Petrobras 3W dataset before designing the loader and anomaly detection models.

**Source:** github.com/petrobras/3W, dataset version 2.0.0 (from `dataset.ini`), cloned to `../3W` next to this project. Data licensed CC BY 4.0.

## 1. Dataset structure and variables

- `dataset/0` to `dataset/9`: one folder per event type. Each Parquet file is one **instance**: a continuous time window from one well.
- File naming: `WELL-00001_20170201010207.parquet` = well 1, starting 2017-02-01 01:02:07. Simulated files start with `SIMULATED`, hand-drawn with `DRAWN`.

Each file has 27 sensors plus `class` and `state`:

| Group | Columns | Units | Notes |
|---|---|---|---|
| Valve states | 9 `ESTADO-*` columns (DHSV, M1, M2, PXO, SDV-GL, SDV-P, W1, W2, XO) | 0, 0.5, 1 | Closed / partial / open; categorical, not continuous |
| Pressures | P-ANULAR, P-JUS-BS, P-JUS-CKGL, P-JUS-CKP, P-MON-CKGL, P-MON-CKP, P-MON-SDV-P, P-PDG, PT-P, P-TPT | Pa | Values in millions (1 bar = 100,000 Pa) |
| Temperatures | T-JUS-CKP, T-MON-CKP, T-PDG, T-TPT | °C | |
| Flow rates | QBS, QGL | m³/s | Service pump and gas lift |
| Choke openings | ABER-CKP, ABER-CKGL | % | Production and gas lift chokes |

Naming: MON = upstream, JUS = downstream; PDG = downhole gauge; TPT = sensor at the subsea Christmas tree.

**Modeling implication:** pressures (millions) and valve states (0–1) are on very different scales, so scaling is required for distance-based and neural models; valve states should be treated as categorical.

## 2. Event types and label scheme

| Label | Event | Transient phase? |
|---|---|---|
| 0 | Normal operation | – |
| 1 | Abrupt increase of BSW | Yes |
| 2 | Spurious closure of DHSV | Yes |
| 3 | Severe slugging | No |
| 4 | Flow instability | No |
| 5 | Rapid productivity loss | Yes |
| 6 | Quick restriction in PCK | Yes |
| 7 | Scaling in PCK | Yes |
| 8 | Hydrate in production line | Yes |
| 9 | Hydrate in service line | Yes |

- `class` labels **each observation**, not each file, so labels can change within an instance.
- `TRANSIENT_OFFSET = 100`: the developing phase of event *k* is labeled *100 + k* (e.g. 106). Expected pattern for transient events: normal (0) → transient (100 + k) → steady event (k).
- The transient phase is the **early-warning window**: detecting an event there gives operators time to act.

### 3W dataset composition (v2.0.0)

- ~2,228 instances: ~1,119 real, ~1,089 simulated, 20 hand-drawn.
- Normal (0) and flow instability (4) are entirely real; several event types are mostly simulated (e.g., class 5: 11 real vs 439 simulated; class 6: 6 real vs 215 simulated).
- Risk: a classifier trained on mixed data could learn "simulated vs real" artifacts instead of physical behavior (shortcut learning), since all normal data is real.

**Decisions**
- Evaluate on real instances only; simulated and drawn instances may be used for training.
- Split by well ID, not by file, to prevent leakage between wells.
- First model: anomaly detection trained on real normal data only, as real examples of several events are too rare for supervised classification.
- Report per-class metrics with the number of real instances behind each.

In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
W3 = ROOT.parent / "3W" / "dataset"   # 3W sits next to assetmind

normal = pd.read_parquet(next((W3 / "0").glob("*.parquet")))
print("Shape:", normal.shape)
print("Index:", normal.index[:3])
print(normal.dtypes)
normal.head()


Shape: (21474, 29)
Index: DatetimeIndex(['2017-02-01 01:02:07', '2017-02-01 01:02:08',
               '2017-02-01 01:02:09'],
              dtype='datetime64[ns]', name='timestamp', freq=None)
ABER-CKGL        float64
ABER-CKP         float64
ESTADO-DHSV      float64
ESTADO-M1        float64
ESTADO-M2        float64
ESTADO-PXO       float64
ESTADO-SDV-GL    float64
ESTADO-SDV-P     float64
ESTADO-W1        float64
ESTADO-W2        float64
ESTADO-XO        float64
P-ANULAR         float64
P-JUS-BS         float64
P-JUS-CKGL       float64
P-JUS-CKP        float64
P-MON-CKGL       float64
P-MON-CKP        float64
P-MON-SDV-P      float64
P-PDG            float64
PT-P             float64
P-TPT            float64
QBS              float64
QGL              float64
T-JUS-CKP        float64
T-MON-CKP        float64
T-PDG            float64
T-TPT            float64
class              Int16
state              Int16
dtype: object


,ABER-CKGL,ABER-CKP,ESTADO-DHSV,ESTADO-M1,ESTADO-M2,ESTADO-PXO,ESTADO-SDV-GL,ESTADO-SDV-P,ESTADO-W1,ESTADO-W2,...,PT-P,P-TPT,QBS,QGL,T-JUS-CKP,T-MON-CKP,T-PDG,T-TPT,class,state
timestamp,,,,,,,,,,,,,,,,,,,,,
2017-02-01 01:02:07,NaN,NaN,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,...,NaN,10074540.0,NaN,0.0,84.64463,NaN,0.0,119.0781,<NA>,<NA>
2017-02-01 01:02:08,NaN,NaN,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,...,NaN,10074540.0,NaN,0.0,84.63828,NaN,0.0,119.0781,<NA>,<NA>
2017-02-01 01:02:09,NaN,NaN,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,...,NaN,10074540.0,NaN,0.0,84.63194,NaN,0.0,119.0781,<NA>,<NA>
2017-02-01 01:02:10,NaN,NaN,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,...,NaN,10074540.0,NaN,0.0,84.62558,NaN,0.0,119.0781,<NA>,<NA>
2017-02-01 01:02:11,NaN,NaN,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,...,NaN,10074540.0,NaN,0.0,84.61923,NaN,0.0,119.0781,<NA>,<NA>


In [2]:
event = pd.read_parquet(next((W3 / "3").glob("WELL-*.parquet")))
print("Columns:", event.columns.tolist())
for col in ["class", "state"]:
    if col in event.columns:
        print(f"\n{col} value counts:\n", event[col].value_counts(dropna=False))

Columns: ['ABER-CKGL', 'ABER-CKP', 'ESTADO-DHSV', 'ESTADO-M1', 'ESTADO-M2', 'ESTADO-PXO', 'ESTADO-SDV-GL', 'ESTADO-SDV-P', 'ESTADO-W1', 'ESTADO-W2', 'ESTADO-XO', 'P-ANULAR', 'P-JUS-BS', 'P-JUS-CKGL', 'P-JUS-CKP', 'P-MON-CKGL', 'P-MON-CKP', 'P-MON-SDV-P', 'P-PDG', 'PT-P', 'P-TPT', 'QBS', 'QGL', 'T-JUS-CKP', 'T-MON-CKP', 'T-PDG', 'T-TPT', 'class', 'state']

class value counts:
 class
3       17976
<NA>     3600
Name: count, dtype: Int64

state value counts:
 state
0       17976
<NA>     3600
Name: count, dtype: Int64


## 4. Inside one real instance (class 3, severe slugging)

- 29 columns: 27 sensors + `class` + `state`. The timestamp is the DataFrame **index** (confirmed), not a column.
- Labels: 17,976 rows labeled `3`, 3,600 rows unlabeled (`<NA>`). No `103` rows, consistent with slugging having no transient phase.
- `state` is missing on the same 3,600 rows; otherwise `0`. Meaning of state codes still to be confirmed from 3W documentation.
- Labels use pandas nullable `Int64` (shows missing as `<NA>`), which the loader must handle.

**Decisions**
- Exclude unlabeled rows from training and evaluation.
- Hypothesis to verify: 1-second sampling, making the unlabeled block exactly one hour.

In [3]:
def label_runs(df: pd.DataFrame) -> pd.DataFrame:
    lab = df["class"].astype("float").fillna(-1)          # -1 marks unlabeled
    run_id = lab.ne(lab.shift()).cumsum()
    return lab.groupby(run_id).agg(label="first", rows="size").reset_index(drop=True)

def describe_instance(path):
    df = pd.read_parquet(path)
    print(f"\n=== {path.parent.name}/{path.name} ===")
    print("Start:", df.index[0], " End:", df.index[-1])
    print("Time steps:", pd.Series(df.index).diff().value_counts().head(3).to_dict())
    print("Label sequence (-1 = unlabeled):\n", label_runs(df).to_string(index=False))
    missing = df.drop(columns=["class", "state"]).isna().mean()
    print("Fully empty sensors:", missing[missing == 1].index.tolist())

describe_instance(next((W3 / "3").glob("WELL-*.parquet")))   # slugging: no transient
describe_instance(next((W3 / "6").glob("WELL-*.parquet")))   # quick PCK restriction: has transient



=== 3/WELL-00001_20170320120025.parquet ===
Start: 2017-03-20 12:00:25  End: 2017-03-20 18:00:00
Time steps: {Timedelta('0 days 00:00:01'): 21575}
Label sequence (-1 = unlabeled):
  label  rows
  -1.0  3600
   3.0 17976
Fully empty sensors: ['ABER-CKGL', 'ABER-CKP', 'P-JUS-BS', 'P-JUS-CKP', 'P-MON-CKGL', 'P-MON-SDV-P', 'PT-P', 'QBS', 'T-MON-CKP']

=== 6/WELL-00002_20140212160333.parquet ===
Start: 2014-02-12 16:03:33  End: 2014-02-12 23:49:42
Time steps: {Timedelta('0 days 00:00:01'): 27969}
Label sequence (-1 = unlabeled):
  label  rows
  -1.0  3600
   0.0 15017
 106.0  2485
   6.0  6868
Fully empty sensors: ['ABER-CKGL', 'ABER-CKP', 'ESTADO-DHSV', 'ESTADO-M1', 'ESTADO-M2', 'ESTADO-PXO', 'ESTADO-SDV-GL', 'ESTADO-SDV-P', 'ESTADO-W1', 'ESTADO-W2', 'ESTADO-XO', 'P-JUS-BS', 'P-JUS-CKGL', 'P-JUS-CKP', 'P-MON-CKGL', 'P-MON-CKP', 'P-MON-SDV-P', 'PT-P', 'QBS', 'QGL', 'T-JUS-CKP', 'T-MON-CKP', 'T-PDG']


## 5. Timelines inside instances

- Sampling is 1 Hz with no gaps in the inspected files.
- Both inspected files start with exactly 3,600 unlabeled rows (1 hour) before labels begin. To verify across all real files.
- Transient event example (class 6, well 2): normal (~4 h 10 min) → transient `106` (~41 min) → event `6` (~1 h 55 min). The transient phase gives a ~40-minute early-warning window.
- Non-transient example (class 3, well 1): unlabeled → event `3` directly, no transient phase.
- Sensor availability differs by well: well 1 has 18 of 27 sensors; well 2 has only 4 (P-ANULAR, P-PDG, P-TPT, T-TPT). Well 2's choke-restriction event must be detected indirectly, as no choke sensors are installed.

**Decision:** measure sensor availability across all real instances before choosing the model's input sensors.

In [4]:
rows = []
for path in sorted(W3.glob("[0-9]/WELL-*.parquet")):
    df = pd.read_parquet(path)
    unlabeled = df["class"].isna()
    present = df.drop(columns=["class", "state"]).notna().any()
    rows.append({
        "class": int(path.parent.name),
        "well": path.name.split("_")[0],
        "file": path.name,
        "n_rows": len(df),
        "n_unlabeled": int(unlabeled.sum()),
        "unlabeled_first_hour": bool(unlabeled.iloc[:3600].all()),
        **present.to_dict(),
    })

scan = pd.DataFrame(rows)
sensor_cols = [c for c in scan.columns
               if c not in {"class", "well", "file", "n_rows", "n_unlabeled", "unlabeled_first_hour"}]
(ROOT / "reports").mkdir(exist_ok=True)
scan.to_parquet(ROOT / "reports/3w_real_scan.parquet", index=False)

print("Real files:", len(scan), "| Distinct wells:", scan["well"].nunique())
print("Files starting with 1 unlabeled hour:", int(scan["unlabeled_first_hour"].sum()))
print("\nShare of real files where each sensor exists:")
print(scan[sensor_cols].mean().sort_values(ascending=False).round(2).to_string())

Real files: 1119 | Distinct wells: 40
Files starting with 1 unlabeled hour: 1119

Share of real files where each sensor exists:
T-TPT            0.92
P-TPT            0.86
P-ANULAR         0.82
P-PDG            0.74
P-MON-CKP        0.65
P-JUS-CKGL       0.62
ESTADO-SDV-P     0.59
T-PDG            0.50
T-JUS-CKP        0.50
ESTADO-W1        0.50
ESTADO-XO        0.48
ESTADO-W2        0.48
ESTADO-PXO       0.48
ESTADO-M2        0.45
ESTADO-M1        0.45
ESTADO-SDV-GL    0.43
ESTADO-DHSV      0.42
QGL              0.40
ABER-CKP         0.27
ABER-CKGL        0.26
T-MON-CKP        0.01
P-JUS-CKP        0.01
P-MON-CKGL       0.00
P-MON-SDV-P      0.00
P-JUS-BS         0.00
QBS              0.00
PT-P             0.00


## 6. Sensor availability across all real instances

- 1,119 real files from 40 wells (~28 files per well), so splitting must be by well.
- All 1,119 files start with exactly one unlabeled hour; the loader drops unlabeled rows.
- Sensor availability tiers:
  - Core (74–92% of files): T-TPT, P-TPT, P-ANULAR, P-PDG: subsea tree temperature and pressure, annulus pressure, downhole pressure.
  - Common (59–65%): P-MON-CKP, P-JUS-CKGL, ESTADO-SDV-P.
  - About half (40–50%): most valve states, T-PDG, T-JUS-CKP, QGL.
  - Rare (~26%): ABER-CKP, ABER-CKGL.
  - Essentially never present in real data (≤1%): T-MON-CKP, P-MON-SDV-P, P-JUS-BS, QBS, PT-P. Dropped.
- Availability drops sharply after the core four (74% to 65%), a natural cut-off for a core sensor set.

**Decisions**
- Baseline anomaly model uses the core four sensors.
- Richer models may add common-tier sensors with missing-value handling, compared against the core-only baseline.

In [5]:
core = ["T-TPT", "P-TPT", "P-ANULAR", "P-PDG"]
scan["n_core"] = scan[core].sum(axis=1)

print("Files with all 4 core sensors:", round(scan["n_core"].eq(4).mean(), 2))
print("Files with at least 3:        ", round(scan["n_core"].ge(3).mean(), 2))

summary = (scan.assign(has_core=scan["n_core"].eq(4))
               .groupby("class")
               .agg(files=("file", "size"),
                    wells=("well", "nunique"),
                    files_with_core=("has_core", "sum")))
summary

Files with all 4 core sensors: 0.71
Files with at least 3:         0.84


,files,wells,files_with_core
class,,,
0,594,9,453
1,4,3,4
2,22,7,21
3,32,2,32
4,343,7,212
5,11,3,11
6,6,2,6
7,36,6,30
8,14,9,3


## 7. Wells per event type and core-sensor coverage

- 71% of real files have all 4 core sensors; 84% have at least 3.
- Normal operation comes from only 9 wells; several events come from just 2–3 wells (class 3: 2 wells, class 6: 2 wells, class 1: 3 wells with 4 files).
- Class 8 (hydrate in production line): 14 files, but only 3 have the core sensors, so a core-only model would barely see this event. Rare sensors can be essential for specific events.

**Decisions**
- Evaluate with leave-one-well-out: train on all wells but one, test on the held-out well, repeat.
- Drop only sensors with essentially no data; measure the value of rare sensors by comparing core-only vs extended models on held-out wells.
- Watch for shortcut learning via sensor-availability fingerprints, since events from few wells could be recognized by their sensor set rather than their behavior. Leave-one-well-out exposes this.
- Use per-instance baselines (deviation from each file's own normal period), mirroring the C-MAPSS baseline-deviation feature, so wells with no normal data in training are not flagged by default.
- Report detection per class with the number of real files and wells behind each result.

In [6]:
import numpy as np

def first_vs_second_hour(cls, file):
    df = pd.read_parquet(W3 / str(cls) / file)[core]
    ratio = df.iloc[:3600].std() / df.iloc[3600:7200].std()
    return ratio.replace([np.inf, -np.inf], np.nan).median()

sample = scan[scan["n_core"].eq(4)].sample(50, random_state=0)
ratios = pd.Series([first_vs_second_hour(c, f) for c, f in zip(sample["class"], sample["file"])])
print(ratios.describe().round(2))

count    50.00
mean      1.16
std       0.82
min       0.00
25%       0.76
50%       0.99
75%       1.19
max       5.45
dtype: float64


## 8. Is the unlabeled first hour a warm-up period?

**Hypothesis:** the first hour is unlabeled because readings fluctuate while the well stabilizes.

**Test:** for 50 random files with all core sensors, compared the standard deviation of each core sensor in hour 1 vs hour 2 (median ratio per file).

**Result:** median ratio 0.99, interquartile range 0.76–1.19. The first hour is no more variable than the second. Mean 1.16 is inflated by a few outliers (max 5.45).

**Conclusion:** hypothesis rejected. The identical 3,600-row length across all 1,119 files also points to a dataset convention rather than physics. The hour is still excluded, since its true state is unknown.

**Data quality flags:** a ratio of 0.00 indicates a frozen (flat) sensor in at least one file; a ratio of 5.45 indicates unusual activity before labeling starts. Both are candidates for automated data quality checks.

## Summary: decisions for the 3W pipeline

**Data scope**
1. Evaluate on real instances only (1,119 files, 40 wells). Simulated and drawn instances may be used for training only.
2. Drop unlabeled rows (always includes the first hour of every file).
3. Drop sensors essentially never present in real data: T-MON-CKP, P-MON-SDV-P, P-JUS-BS, QBS, PT-P.

**Sensors**
4. Core set for the baseline model: T-TPT, P-TPT, P-ANULAR, P-PDG (all four present in 71% of real files).
5. Extended models may add other sensors with missing-value handling; their value is measured against the core-only baseline on held-out wells.
6. Valve states are categorical (0 / 0.5 / 1); pressures need scaling (Pa, values in millions).

**Labels**
7. Labels are per row: normal 0, transient 100 + k, event k. The transient phase is the early-warning window.
8. Severe slugging (3) and flow instability (4) have no transient phase.

**Evaluation**
9. Leave-one-well-out evaluation; files from the same well never appear in both training and testing.
10. Report detection per class with the number of real files and wells behind each result.
11. Watch for shortcut learning (simulated vs real artifacts, sensor-availability fingerprints).

**Features**
12. Per-instance baseline deviation, measured against each file's own normal period, mirroring the C-MAPSS baseline feature.

**Data quality**
13. Flag frozen sensors (zero variance) and unusual pre-label activity.